In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
data = pd.read_csv('Telco-Customer-Churn-Cleaned.csv')

In [4]:
print(data.head())

   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       0              0        1           0       1             0   
1       1              0        0           0      34             1   
2       1              0        0           0       2             1   
3       1              0        0           0      45             0   
4       0              0        0           0       2             1   

   PaperlessBilling  MonthlyCharges TotalCharges  Churn  ...  \
0                 1           29.85        29.85      0  ...   
1                 0           56.95       1889.5      0  ...   
2                 1           53.85       108.15      1  ...   
3                 0           42.30      1840.75      0  ...   
4                 1           70.70       151.65      1  ...   

   StreamingMovies_No  StreamingMovies_No internet service  \
0                   1                                    0   
1                   1                                    0   
2 

In [5]:
data.replace('', np.nan, inplace=True)

In [6]:
data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
data['TotalCharges'].fillna(data['TotalCharges'].median(), inplace=True)

In [7]:
X = data.drop(columns=['Churn'])  
y = data['Churn']  

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
rf_model = RandomForestClassifier(random_state=42, n_estimators=100)

rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [10]:
y_pred = rf_model.predict(X_test)

In [11]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy for Random Forest: {accuracy:.2f}")

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix)

Accuracy for Random Forest: 0.78
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.88      0.86      1035
           1       0.61      0.51      0.55       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.71      1409
weighted avg       0.77      0.78      0.78      1409

Confusion Matrix:
[[915 120]
 [185 189]]


In [12]:
feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(feature_importances)

                                    Feature  Importance
8                              TotalCharges    0.167951
7                            MonthlyCharges    0.147770
4                                    tenure    0.144124
33                  Contract_Month-to-month    0.064345
24                           TechSupport_No    0.032425
15                        OnlineSecurity_No    0.028230
13              InternetService_Fiber optic    0.028190
0                                    gender    0.028040
38           PaymentMethod_Electronic check    0.027481
6                          PaperlessBilling    0.024784
2                                   Partner    0.023002
1                             SeniorCitizen    0.020893
3                                Dependents    0.019204
18                          OnlineBackup_No    0.015138
35                        Contract_Two year    0.015034
21                      DeviceProtection_No    0.014695
20                         OnlineBackup_Yes    0

In [13]:
from sklearn.model_selection import GridSearchCV

# Define hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10]
}

# Initialize Grid Search
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

# Best parameters
print("Best Parameters:", grid_search.best_params_)

# Best model
best_rf_model = grid_search.best_estimator_

Best Parameters: {'max_depth': 20, 'min_samples_split': 10, 'n_estimators': 200}
